In [0]:
%sql
create catalog catalog_durga;

In [0]:
%sql
create schema catalog_durga.source;

#unmanaged location
# will have names as given unlike some table id in managed location.

In [0]:
%sql
drop catalog if exists hr; create catalog hr MANAGED LOCATION 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/'

In [0]:
%sql
create schema hr.bronze managed location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/bronze';
create schema hr.silver managed location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/silver';
create schema hr.gold managed location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/gold';

In [0]:
%sql
create table hr.bronze.emp (id int, name string) using delta location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/bronze/emp'

In [0]:
%sql
create external volume hr.bronze.source location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/bronze/source'

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8935170284305541>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create external volume hr.bronze.source location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/bronze/source'\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, lin

In [0]:
%sql
insert into hr.bronze.emp values(1,'John'),(2,'Mary');

num_affected_rows,num_inserted_rows
2,2


In [0]:
# Want to replace existing data in table with Dataframe.

from pyspark.sql.functions import cast
from pyspark.sql.types import IntegerType
Ls=[(1,'naveen'),(2,'durga')]
sc=["id","name"]
df=spark.createDataFrame(data=Ls,schema=sc)
ds=df.withColumn("id",df.id.cast(IntegerType()))
ds.show()

ds1= spark.read.format("Delta").table("hr.bronze.emp")
ds1.show()

ds.write.mode("overwrite").format("delta").saveAsTable("hr.bronze.emp")

ds2= spark.read.format("Delta").table("hr.bronze.emp")
ds2.show()

+---+------+
| id|  name|
+---+------+
|  1|naveen|
|  2| durga|
+---+------+

+---+----+
| id|name|
+---+----+
|  1|John|
|  2|Mary|
+---+----+

+---+------+
| id|  name|
+---+------+
|  1|naveen|
|  2| durga|
+---+------+



In [0]:
#want to append dataframe to existing table.

from pyspark.sql.functions import cast
from pyspark.sql.types import IntegerType

ls=[(1,"naveen"),(2,"durga")]
df=spark.createDataFrame(ls,["ID","Name"])
ds=df.withColumn("id",df.ID.cast(IntegerType()))
ds.show()

ds1=spark.read.format("delta").table("hr.bronze.emp")
ds1.show()

ds.write.mode("append").format("delta").saveAsTable("hr.bronze.emp")

ds2=spark.read.format("delta").table("hr.bronze.emp")
ds2.show()

+---+------+
| id|  Name|
+---+------+
|  1|naveen|
|  2| durga|
+---+------+

+---+------+
| id|  name|
+---+------+
|  1|naveen|
|  2| durga|
|  1|naveen|
|  2| durga|
+---+------+

+---+------+
| id|  name|
+---+------+
|  1|naveen|
|  2| durga|
|  1|naveen|
|  2| durga|
|  1|naveen|
|  2| durga|
+---+------+



In [0]:
# DML can be done in delta instance created based on delta table as delta table can't perform dml directly on it.

#it can be done in 2 ways . 
#METHOD#1: for name method
    #syntax: DeltaTable.forName(spark, '<delta_table_name>')

#method# 1: for path method
    #syntax:-  DeltaTable.forPath(spark, '<ADLS LOCATION>')

In [0]:
from pyspark.sql.functions import cast
from pyspark.sql.types import IntegerType

ls=[(4,"bhavani"),(2,"sri durga")]
df=spark.createDataFrame(ls,["ID","Name"])
ds=df.withColumn("id",df.ID.cast(IntegerType()))
ds.show()

ds1=spark.read.format("delta").table("hr.bronze.emp")
ds1.show()

from delta.tables import *
del_ins=DeltaTable.forName(spark,"hr.bronze.emp")

del_ins.alias("t").merge(
    ds.alias("s"), condition = "s.ID == t.ID" ).whenMatchedUpdate( set =
                     {
                         'Name': "s.Name"
                     }

    ).whenNotMatchedInsert(values =
                        {
                            "ID":"s.ID",
                            "Name":"s.Name"
                        }

    ).execute()

ds2=spark.read.format("delta").table("hr.bronze.emp")
ds2.show()

+---+---------+
| id|     Name|
+---+---------+
|  4|  bhavani|
|  2|sri durga|
+---+---------+

+---+------+
| id|  name|
+---+------+
|  1|naveen|
|  2| durga|
|  1|naveen|
|  2| durga|
|  1|naveen|
|  2| durga|
+---+------+

+---+---------+
| id|     name|
+---+---------+
|  2|sri durga|
|  2|sri durga|
|  2|sri durga|
|  1|   naveen|
|  1|   naveen|
|  1|   naveen|
|  4|  bhavani|
+---+---------+



In [0]:
%sql
create table hr.bronze.scd_dest(EMPID INT, NAME STRING, DESIGNATON STRING, DEPARTMENT STRING, START_DATE DATE, END_DATE DATE, ACTIVE_FLAG STRING) using delta location 'abfss://ex-metastore@meadls.dfs.core.windows.net/root/HR/bronze/scd_dest'

In [0]:
%sql
Insert into hr.bronze.scd_dest VALUES
(101, 'ADAM', 'SALES ASSOCIATE', 'SALES', '2021-01-01', '2021-12-31', 'N'),
(101, 'ADAM', 'SALES OFFICER', 'SALES', '2022-01-01', '9999-12-31', 'Y'),
(102, 'JASON', 'SALES OFFICER', 'SALES', '2022-01-01', '9999-12-31', 'Y'),
(200, 'JACOB', 'SALES OFFICER', 'SALES', '2022-01-01', '9999-12-31', 'Y');

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
select * from hr.bronze.scd_dest version as of 1;
select * from hr.bronze.scd_dest timestamp as of "2026-04-17 10:25:25.0";

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6569652547706746>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from hr.bronze.scd_dest version as of 1;\nselect * from hr.bronze.scd_dest timestamp as of "2026-04-17 10:25:25.0";\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, 

In [0]:
%sql
select * from hr.bronze.scd_dest;

EMPID,NAME,DESIGNATON,DEPARTMENT,START_DATE,END_DATE,ACTIVE_FLAG
101,ADAM,SALES ASSOCIATE,SALES,2021-01-01,2021-12-31,N
101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
102,JASON,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
200,JACOB,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y


In [0]:
list = [(101,"ADAM","TEAM LEADER","SALES","2024-12-02","9999-12-31"),(102,"JASON","SALES OFFICER","SALES","2022-01-01","9999-12-31"),(103,"JESS","MANAGER","SALES","2024-12-02","9999-12-31")]
cols = ["EMPID","NAME","DESIGNATION","DEPARTMENT","START_DATE","END_DATE"]
source_df=spark.createDataFrame(list,cols)
source_df.show()

dest_df=spark.read.format("delta").table("hr.bronze.scd_dest")
dest_df.show()




+-----+-----+-------------+----------+----------+----------+
|EMPID| NAME|  DESIGNATION|DEPARTMENT|START_DATE|  END_DATE|
+-----+-----+-------------+----------+----------+----------+
|  101| ADAM|  TEAM LEADER|     SALES|2024-12-02|9999-12-31|
|  102|JASON|SALES OFFICER|     SALES|2022-01-01|9999-12-31|
|  103| JESS|      MANAGER|     SALES|2024-12-02|9999-12-31|
+-----+-----+-------------+----------+----------+----------+

+-----+-----+---------------+----------+----------+----------+-----------+
|EMPID| NAME|     DESIGNATON|DEPARTMENT|START_DATE|  END_DATE|ACTIVE_FLAG|
+-----+-----+---------------+----------+----------+----------+-----------+
|  101| ADAM|SALES ASSOCIATE|     SALES|2021-01-01|2021-12-31|          N|
|  101| ADAM|  SALES OFFICER|     SALES|2022-01-01|9999-12-31|          Y|
|  102|JASON|  SALES OFFICER|     SALES|2022-01-01|9999-12-31|          Y|
|  200|JACOB|  SALES OFFICER|     SALES|2022-01-01|9999-12-31|          Y|
+-----+-----+---------------+----------+-------

In [0]:
join_df=source_df.join(dest_df,source_df.EMPID==dest_df.EMPID,"left") .select (source_df['*'], dest_df.EMPID.alias("dest_empid"),dest_df.NAME.alias("dest_name"),dest_df.DESIGNATON.alias("dest_designation"),dest_df.DEPARTMENT.alias("dest_depart"),dest_df.START_DATE.alias("dest_start"),dest_df.END_DATE.alias("dest_end"),dest_df.ACTIVE_FLAG.alias("dest_flag"))
display(join_df)


EMPID,NAME,DESIGNATION,DEPARTMENT,START_DATE,END_DATE,dest_empid,dest_name,dest_designation,dest_depart,dest_start,dest_end,dest_flag
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES ASSOCIATE,SALES,2021-01-01,2021-12-31,N
102,JASON,SALES OFFICER,SALES,2022-01-01,9999-12-31,102,JASON,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
103,JESS,MANAGER,SALES,2024-12-02,9999-12-31,null,null,null,null,null,null,null


In [0]:
f1=join_df.filter("dest_flag == 'Y' and START_DATE <> dest_start and DESIGNATION <> dest_designation")
display(f1)

f2=join_df.filter("dest_flag is NULL")
display(f2)

f3=f1.union(f2)
display(f3)



EMPID,NAME,DESIGNATION,DEPARTMENT,START_DATE,END_DATE,dest_empid,dest_name,dest_designation,dest_depart,dest_start,dest_end,dest_flag
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y


EMPID,NAME,DESIGNATION,DEPARTMENT,START_DATE,END_DATE,dest_empid,dest_name,dest_designation,dest_depart,dest_start,dest_end,dest_flag
103,JESS,MANAGER,SALES,2024-12-02,9999-12-31,null,null,null,null,null,null,null


EMPID,NAME,DESIGNATION,DEPARTMENT,START_DATE,END_DATE,dest_empid,dest_name,dest_designation,dest_depart,dest_start,dest_end,dest_flag
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
103,JESS,MANAGER,SALES,2024-12-02,9999-12-31,null,null,null,null,null,null,null


In [0]:
from pyspark.sql.functions import concat,lit
f4=f3.withColumn("matchkey",concat("EMPID",lit("A")))
f5=f1.withColumn("matchkey",concat("EMPID",lit("Y")))
final=f4.union(f5)

display(final)


EMPID,NAME,DESIGNATION,DEPARTMENT,START_DATE,END_DATE,dest_empid,dest_name,dest_designation,dest_depart,dest_start,dest_end,dest_flag,matchkey
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y,101A
103,JESS,MANAGER,SALES,2024-12-02,9999-12-31,null,null,null,null,null,null,null,103A
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,101,ADAM,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y,101Y


In [0]:
from delta.tables import DeltaTable
delta_inst=DeltaTable.forName(spark,"hr.bronze.scd_Dest")
display(delta_inst)



In [0]:
delta_inst.alias("T").merge(
  final.alias("S"),
  condition = "Concat(T.EMPID,T.ACTIVE_FLAG) == S.matchkey and T.ACTIVE_FLAG ='Y'"
  ).whenMatchedUpdate (set =
                     {
                       "END_DATE" : "S.START_DATE",
                       "ACTIVE_FLAG" : "'N'"
                     }
                     
  ).whenNotMatchedInsert (values = 
                       {
                        "EMPID" : "S.EMPID",
                        "NAME" :"S.NAME",
                        "DESIGNATON" : "S.DESIGNATION",
                        "DEPARTMENT" : "S.DEPARTMENT",
                        "START_DATE" : "S.START_DATE",
                        "END_DATE" : "S.END_DATE",
                        "ACTIVE_FLAG" : "'Y'" 
                       }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df = spark.read.format("delta").table("hr.bronze.scd_dest")
df.show()

+-----+-----+---------------+----------+----------+----------+-----------+
|EMPID| NAME|     DESIGNATON|DEPARTMENT|START_DATE|  END_DATE|ACTIVE_FLAG|
+-----+-----+---------------+----------+----------+----------+-----------+
|  101| ADAM|SALES ASSOCIATE|     SALES|2021-01-01|2021-12-31|          N|
|  102|JASON|  SALES OFFICER|     SALES|2022-01-01|9999-12-31|          Y|
|  200|JACOB|  SALES OFFICER|     SALES|2022-01-01|9999-12-31|          Y|
|  101| ADAM|    TEAM LEADER|     SALES|2024-12-02|9999-12-31|          Y|
|  103| JESS|        MANAGER|     SALES|2024-12-02|9999-12-31|          Y|
|  101| ADAM|  SALES OFFICER|     SALES|2022-01-01|2024-12-02|          N|
+-----+-----+---------------+----------+----------+----------+-----------+



In [0]:
%sql
select * from hr.bronze.scd_dest version as of 2;

EMPID,NAME,DESIGNATON,DEPARTMENT,START_DATE,END_DATE,ACTIVE_FLAG
101,ADAM,SALES ASSOCIATE,SALES,2021-01-01,2021-12-31,N
102,JASON,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
200,JACOB,SALES OFFICER,SALES,2022-01-01,9999-12-31,Y
101,ADAM,SALES OFFICER,SALES,2022-01-01,2024-12-02,N
101,ADAM,TEAM LEADER,SALES,2024-12-02,9999-12-31,Y
103,JESS,MANAGER,SALES,2024-12-02,9999-12-31,Y


#Time Travel

In [0]:
%sql
create table hr.bronze.timetravel (id int,name string) using delta

In [0]:
%sql

DESCRIBE HISTORY hr.bronze.timetravel;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-17T15:12:26.000Z,145728833949047,me@azuremaster276gmail.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3462144281410412),3dd042fe-33d8-47fd-b900-03cf217f1df7,0417-141013-e9vkl4ho-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 785)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-04-17T15:12:25.000Z,145728833949047,me@azuremaster276gmail.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3462144281410412),b51390ec-3e0b-453c-9110-b39f2f4e5e3f,0417-141013-e9vkl4ho-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 785)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-04-17T15:10:07.000Z,145728833949047,me@azuremaster276gmail.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-631ae623-a296-42eb-8d43-81ec8a9dc7db"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-b5cf211a-07cc-4c6c-9a0e-5bb5bfe26c3f""}, statsOnLoad -> false)",null,List(3462144281410412),e8530642-b81a-4eba-8ffd-e6261baf944d,0417-141013-e9vkl4ho-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
%sql
insert into hr.bronze.timetravel values(1,"a");
insert into hr.bronze.timetravel values(2,"e");

num_affected_rows,num_inserted_rows
1,1


In [0]:

%sql

select * from hr.bronze.timetravel VERSION AS OF 1;

  File <command-5474176669134397>, line 4
    SELECT * FROM delta.`hr.bronze.timetravel` VERSION AS OF 1;
                  ^
SyntaxError: invalid syntax


In [0]:
%sql
select * from hr.bronze.timetravel timestamp as of "2026-04-17T15:12:25.000+00:00";

id,name
1,a


In [0]:
%sql
--restore to previous version of data
restore table  hr.bronze.timetravel timestamp as of "2026-04-17T15:12:25.000+00:00";

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
785,1,1,0,785,0


In [0]:
%sql
select * from hr.bronze.timetravel;

id,name
1,a



#Time Travel in Pyspark

In [0]:
df=spark.read.format("delta").option("versionAsOf",2).table("hr.bronze.timetravel")
df.show()

+---+----+
| id|name|
+---+----+
|  2|   e|
|  1|   a|
+---+----+



In [0]:
df=spark.read.format("delta").option("timestampAsof","2026-04-17T15:12:25.000+00:00").table("hr.bronze.timetravel")
df.show()

+---+----+
| id|name|
+---+----+
|  1|   a|
+---+----+



In [0]:
#create deltainstance for delta table to restore data in delta table.
from delta.tables import DeltaTable
dt=DeltaTable.forName(spark,"hr.bronze.timetravel")
dt.restoreToVersion(1)

ds=spark.read.format("delta").table("hr.bronze.timetravel")
display(ds)


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-6853008411884973>, line 6
      3 dt=DeltaTable.forName(spark,"hr.bronze.timetravel")
      4 dt.restoreToVersion(1)
----> 6 ds=spark.read.format("delta").tables(hr.bronze.timetravel)
      7 display(ds)

AttributeError: 'DataFrameReader' object has no attribute 'tables'

In [0]:
#create deltainstance for delta table to restore data in delta table.
from delta.tables import DeltaTable
dt=DeltaTable.forName(spark,"hr.bronze.timetravel")
dt.restoreToTimestamp("2026-04-17T15:12:25.000+00:00")

ds=spark.read.format("delta").table("hr.bronze.timetravel")
display(ds)


id,name
1,a


In [0]:
%sql

Describe History hr.bronze.timetravel;